In [ ]:
!pip install joblib
!pip install scikit-learn
!pip install imbalanced-learn
!pip install numpy
!pip install pandas
!pip install scikit-learn
!pip install imbalanced-learn
!pip install joblib
!pip install scikit-learn-intelex
!pip install deap
!pip install pyswarms

In [ ]:
# Use on Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Intel Extension for Scikit-learn (formerly called scikit-learn-intelex).
# It replaces certain algorithms in scikit-learn with faster Intel-optimized implementations.
from sklearnex import patch_sklearn
patch_sklearn()

In [ ]:
import joblib

datasets = joblib.load('datasets.jbl')

# sum(datasets['X_russian']['I3']) is 9, almost all entries are 0; I3 is unreliable supplier
# taiwanese data has a constant column - this triggers a warning, which is harmless

countries = ["polish", "russian", "taiwanese"]

country = countries[2]
X = datasets[f'X_{country}']
y = datasets[f'y_{country}'].squeeze() # converts 1d dataframe to series

if country == "polish":
    # Select rows where year == 5.0
    mask = X["year"] == 5.0
    X = X[mask].drop(columns=["year"])
    y = y[mask]

N = len(X.columns)

In [ ]:
# === Cross-validation setup ===
from sklearn.model_selection import StratifiedKFold, ShuffleSplit
import joblib

cv1 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

d = {
    '10 fold splits': [(train_idx, test_idx) for train_idx, test_idx in cv1.split(X, y)],
}

joblib.dump(d, f'cvs_{country}.jbl')

In [ ]:
#
# Genetic Algorithm (GA)
#

import numpy as np
import random

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

from deap import base, creator, tools, algorithms


class GeneticFeatureSelector(BaseEstimator, TransformerMixin):
    """
    A scikit-learn compatible transformer that performs feature selection
    using a Genetic Algorithm (via the DEAP library).
    """
    def __init__(self, n_features_to_select=5, population_size=100, generations=50,
                 patience=10, cv=5, random_state=42, verbose=True):

        # Number of features the GA must select
        self.n_features_to_select = n_features_to_select

        # Number of candidate solutions (feature subsets) per generation
        self.population_size = population_size

        # Maximum number of generations the GA will run
        self.generations = generations

        # Early stopping: stop if no improvement after this many generations
        self.patience = patience

        # Cross-validation folds used to evaluate a feature subset
        self.cv = cv

        # Random seed for reproducibility
        self.random_state = random_state

        # Whether to print progress information
        self.verbose = verbose


    # ---------------------------------------------------------
    # INITIALIZE DEAP COMPONENTS (GENETIC ALGORITHM SETUP)
    # ---------------------------------------------------------
    def _init_deap(self):

        # Fix random seed for reproducibility
        random.seed(self.random_state)

        # Total number of available features in dataset
        self.n_total_features = self.X_.shape[1]

        # DEAP uses a global "creator" registry.
        # We ensure these objects are created only once.
        if not hasattr(self, '_creator_initialized'):

            # Define a fitness type that should be maximized
            creator.create("FitnessMax", base.Fitness, weights=(1.0,))

            # Define an Individual = list with attached FitnessMax
            creator.create("Individual", list, fitness=creator.FitnessMax)

            self._creator_initialized = True

        # Toolbox is DEAP’s container for genetic operators
        self.toolbox = base.Toolbox()


        # ---------------------------------------------------------
        # INDIVIDUAL CREATION
        # ---------------------------------------------------------
        def init_individual():

            # Individual = binary list indicating feature selection
            # Example: [0,1,0,1,0] means select features 1 and 3
            ind = [0] * self.n_total_features

            # Randomly choose exactly n_features_to_select positions
            selected = random.sample(
                range(self.n_total_features), 
                self.n_features_to_select
            )

            # Set those positions to 1
            for i in selected:
                ind[i] = 1

            # Convert to DEAP Individual object
            return creator.Individual(ind)

        # Register function that creates a single individual
        self.toolbox.register("individual", init_individual)

        # Register function that creates a population (list of individuals)
        self.toolbox.register("population", tools.initRepeat, list, self.toolbox.individual)


        # ---------------------------------------------------------
        # FITNESS CACHE
        # ---------------------------------------------------------
        # Avoid re-evaluating identical feature subsets
        self.fitness_cache = {}


        # ---------------------------------------------------------
        # FITNESS FUNCTION
        # ---------------------------------------------------------
        def evaluate(ind):

            # Convert individual to tuple so it can be used as dictionary key
            key = tuple(ind)

            # If already evaluated before, reuse cached score
            if key in self.fitness_cache:
                return self.fitness_cache[key]

            # Determine which features are selected (bit == 1)
            selected = [i for i, bit in enumerate(ind) if bit == 1]

            # If no features selected -> invalid solution
            if len(selected) == 0:
                return 0.,

            # Train logistic regression on only selected features
            # and evaluate using cross-validation
            score = cross_val_score(
                LogisticRegression(
                    solver='liblinear',
                    max_iter=2000,
                    random_state=self.random_state
                ),
                self.X_[:, selected],   # subset of features
                self.y_,
                cv=self.cv
            ).mean()

            # Store in cache
            self.fitness_cache[key] = (score,)

            return score,

        # Register evaluation function
        self.toolbox.register("evaluate", evaluate)


        # ---------------------------------------------------------
        # CROSSOVER (MATING)
        # ---------------------------------------------------------
        def mate(ind1, ind2):

            # Perform two-point crossover between parents
            tools.cxTwoPoint(ind1, ind2)

            # Ensure children still select exactly n_features_to_select
            for ind in [ind1, ind2]:

                while sum(ind) != self.n_features_to_select:

                    # Indices of selected features
                    ones = [i for i, x in enumerate(ind) if x == 1]

                    # Indices of unselected features
                    zeros = [i for i, x in enumerate(ind) if x == 0]

                    # Too many features selected -> turn one off
                    if sum(ind) > self.n_features_to_select:
                        ind[random.choice(ones)] = 0

                    # Too few features selected -> turn one on
                    elif sum(ind) < self.n_features_to_select:
                        ind[random.choice(zeros)] = 1

            return ind1, ind2


        # ---------------------------------------------------------
        # MUTATION
        # ---------------------------------------------------------
        def mutate(ind):

            # Find positions with selected features
            ones = [i for i, bit in enumerate(ind) if bit == 1]

            # Find positions with unselected features
            zeros = [i for i, bit in enumerate(ind) if bit == 0]

            # Swap one selected feature with one unselected feature
            if ones and zeros:
                i1, i0 = random.choice(ones), random.choice(zeros)
                ind[i1], ind[i0] = 0, 1

            return ind,


        # Register genetic operators
        self.toolbox.register("mate", mate)
        self.toolbox.register("mutate", mutate)

        # Selection method: tournament selection
        self.toolbox.register("select", tools.selTournament, tournsize=3)



    # ---------------------------------------------------------
    # TRAINING PHASE (RUN GENETIC ALGORITHM)
    # ---------------------------------------------------------
    def fit(self, X, y):

        # Store training data as numpy arrays
        self.X_ = np.array(X)
        self.y_ = np.array(y)

        # Initialize DEAP toolbox and operators
        self._init_deap()

        # Create initial random population
        pop = self.toolbox.population(n=self.population_size)

        # Track best score found
        best_fitness = -np.inf

        # Counter for early stopping
        stagnation = 0


        # Main GA loop (generations)
        for gen in range(1, self.generations + 1):

            # Apply crossover and mutation to produce offspring
            offspring = algorithms.varAnd(pop, self.toolbox, cxpb=0.5, mutpb=0.2)

            # Identify individuals that have not been evaluated yet
            invalid = [ind for ind in offspring if not ind.fitness.valid]

            # Compute fitness for them
            fitnesses = list(map(self.toolbox.evaluate, invalid))

            # Attach fitness scores
            for ind, fit in zip(invalid, fitnesses):
                ind.fitness.values = fit

            # Select next generation population
            pop = self.toolbox.select(offspring, k=len(pop))

            # Find best individual in population
            best_ind = tools.selBest(pop, 1)[0]
            best_score = best_ind.fitness.values[0]

            # Print progress
            if self.verbose:
                print(f"Generation {gen:>2}: Best CV score = {best_score:.4f}")

            # Check improvement
            if best_score > best_fitness:

                best_fitness = best_score
                self.best_individual_ = best_ind

                stagnation = 0

            else:

                stagnation += 1

                # Early stopping if no improvement
                if stagnation >= self.patience:
                    if self.verbose:
                        print(f"Early stopping at generation {gen} (no improvement for {self.patience} generations).")
                    break


        # Extract indices of selected features from best individual
        self.selected_features_ = [i for i, bit in enumerate(self.best_individual_) if bit == 1]

        # Store best score
        self.best_score_ = best_fitness

        # Boolean mask for sklearn compatibility
        self.support_ = np.zeros(self.n_total_features, dtype=bool)
        self.support_[self.selected_features_] = True

        return self


    # ---------------------------------------------------------
    # APPLY FEATURE SELECTION TO NEW DATA
    # ---------------------------------------------------------
    def transform(self, X):
        # Keep only the selected columns
        return X[:, self.support_]


    # ---------------------------------------------------------
    # RETURN FEATURE MASK
    # ---------------------------------------------------------
    def get_support(self, indices=False):
        """
        Return selected features.

        indices=False -> boolean mask of selected features
        indices=True  -> integer indices of selected features
        """
        return np.where(self.support_)[0] if indices else self.support_

In [ ]:
#
# Particle Swarm Optimization (PSO)
#

import numpy as np

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

import pyswarms as ps

class PSOFeatureSelector(BaseEstimator, TransformerMixin):
    """
    A scikit-learn compatible transformer that performs feature selection
    using Particle Swarm Optimization (PSO).
    """
    def __init__(self, n_select=5, n_particles=20, iters=50,
                 pso_options=None, estimator=None, cv=5, random_state=42):

        # Number of features to select
        self.n_select = n_select

        # Number of particles in the swarm (candidate solutions)
        self.n_particles = n_particles

        # Number of optimization iterations
        self.iters = iters

        # Number of folds used for cross-validation when evaluating a feature subset
        self.cv = cv

        # Random seed for reproducibility
        self.random_state = random_state

        # PSO hyperparameters:
        # c1 = cognitive coefficient (particle's own best influence)
        # c2 = social coefficient (global best influence)
        # w  = inertia weight (momentum of movement)
        self.pso_options = pso_options or {'c1': 1.5, 'c2': 1.5, 'w': 0.7}

        # Base estimator used to evaluate feature subsets
        # Default: Logistic Regression
        self.estimator = estimator or LogisticRegression(
            solver='liblinear',
            max_iter=2000,
            random_state=random_state
        )

        # Stores indices of the selected features after fitting
        self.selected_indices_ = None


    # ---------------------------------------------------------
    # OBJECTIVE FUNCTION FOR PSO
    # ---------------------------------------------------------
    # PSO works by minimizing a cost function. This function evaluates
    # the quality of each particle (candidate feature ranking).
    def _objective_function(self, particles):

        scores = []

        # Each particle represents a vector of feature "weights"
        # Shape: (n_particles, n_features)
        for particle in particles:

            # Rank features according to particle values
            # Select the indices of the top-n features
            top_indices = np.argsort(particle)[-self.n_select:]

            # Convert selected indices to a boolean mask
            mask = np.zeros_like(particle, dtype=bool)
            mask[top_indices] = True

            # Evaluate the selected feature subset using cross-validation
            score = cross_val_score(
                self.estimator,
                self.X_[:, mask],  # use only selected features
                self.y_,
                cv=self.cv
            ).mean()

            # PSO minimizes the objective function,
            # so we negate the score (since higher accuracy is better)
            scores.append(-score)

        # Return cost for each particle
        return np.array(scores)


    # ---------------------------------------------------------
    # FIT METHOD (RUN PSO OPTIMIZATION)
    # ---------------------------------------------------------
    def fit(self, X, y):

        # Store training data
        self.X_ = X
        self.y_ = y

        # Total number of features
        n_features = X.shape[1]

        # Initialize PSO optimizer from pyswarms
        optimizer = ps.single.GlobalBestPSO(

            # Number of particles exploring the search space
            n_particles=self.n_particles,

            # Dimensionality of the problem (one dimension per feature)
            dimensions=n_features,

            # PSO hyperparameters
            options=self.pso_options
        )

        # Run the PSO optimization
        # cost = best objective value found
        # pos  = best particle position (feature weight vector)
        cost, pos = optimizer.optimize(
            self._objective_function,
            iters=self.iters,
            verbose=False
        )

        # After optimization, select the top-n ranked features
        # according to the best particle position
        self.selected_indices_ = np.argsort(pos)[-self.n_select:]

        return self


    # ---------------------------------------------------------
    # APPLY FEATURE SELECTION TO NEW DATA
    # ---------------------------------------------------------
    def transform(self, X):
        # Keep only the selected columns
        return X[:, self.selected_indices_]


    # ---------------------------------------------------------
    # RETURN FEATURE MASK
    # ---------------------------------------------------------
    def get_support(self, indices=False):
        """
        Return selected features.

        indices=False -> boolean mask of selected features
        indices=True  -> integer indices of selected features
        """

        # Create boolean mask of selected features
        mask = np.zeros(self.X_.shape[1], dtype=bool)
        mask[self.selected_indices_] = True

        # Return indices or mask
        return self.selected_indices_ if indices else mask

In [ ]:
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

# Filter methods
score_funcs = [
    ("Anova", f_classif),
    ("Info", mutual_info_classif),
]

# Wrapper, Embedded
fsmethods_embedded = [
    ("LogisticRegressionWitL1Penalty", LogisticRegression(penalty='l1', solver='liblinear', max_iter=2000, random_state=42)),
    ("RandomForest", RandomForestClassifier(random_state=42)),
]

# GA, PSO
fsmethods_wrapper = [
    ("LogisticRegressionWitL1Penalty", LogisticRegression(penalty='l1', solver='liblinear', max_iter=2000, random_state=42)),
    ("RandomForest", RandomForestClassifier(random_state=42)),
    ("PSO", PSOFeatureSelector(n_select=5, iters=30)),
    ("GA", GeneticFeatureSelector(n_features_to_select=5, generations=30, verbose=True)),
]

fsmethods = fsmethods_embedded

# --- Balancing Methods ---
blmethods = [
    ("SMOTE", SMOTE(random_state=42)),
    ("RandomOverSampler", RandomOverSampler(random_state=42)),
    ("RandomUnderSampler", RandomUnderSampler(random_state=42)),
]

# --- Classifiers ---
clmethods = [
    ("RandomForestClassifier", RandomForestClassifier(random_state=42, n_jobs=-1)),
    ("LinearSVC", SVC(kernel='linear', n_jobs=-1)),
    ("SVC", SVC(kernel='rbf', C=1.0, gamma='scale', n_jobs=-1)),
    ("GradientBoostingClassifier", GradientBoostingClassifier(random_state=42)),
    ("KNNClassifier", KNeighborsClassifier(n_jobs=-1)),
    ("NaiveBayes", GaussianNB()),
    ("CART", DecisionTreeClassifier(random_state=42)),
]

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score
from imblearn.metrics import geometric_mean_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import joblib
import os


# Varying the percentiles for feature selection
percentiles = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

# Cross-validation loop
for cvi, (train_idx, test_idx) in enumerate(joblib.load(f'cvs_{country}.jbl')['10 fold splits']):
    cv_name = f"Fold{cvi}"

    Xcopy = X.copy()
    constant_columns = Xcopy.loc[:, Xcopy.nunique() <= 1].columns
    Xcopy = Xcopy.drop(columns=constant_columns)

    N = len(Xcopy.columns)

    X_train, X_test = Xcopy.iloc[train_idx], Xcopy.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    switch = True # (bal+fs)

    for blmethod_name, blmethod in blmethods[:]:
        balancer = blmethod
        X_balanced, y_balanced = balancer.fit_resample(X_train, y_train)

        for fsmethod_name, fsmethod in fsmethods[:]:
            scaler = StandardScaler()
            scaler.fit(X_balanced)
            X_scaled = scaler.transform(X_balanced)
            
            for percentile in percentiles[:]:
                pct_name = f"Pct{percentile}"

                n = int(np.ceil(percentile / 100. * N))

                if fsmethod_name in ["LogisticRegressionWitL1Penalty", "RandomForest"]:
                    model = fsmethod
                    selector = RFE(estimator=model, n_features_to_select=n, step=5)
                elif fsmethod_name == "PSO":
                    selector = PSOFeatureSelector(n_select=n)
                elif fsmethod_name == "GA":
                    selector = GeneticFeatureSelector(n_features_to_select=n)

                selector.fit(X_scaled, y_balanced)
                mask = selector.support_
                X_selected = X_scaled[:, mask]
    
                for clmethod_name, clmethod in clmethods[:]:
                    clf = clmethod
                
                    # Step 4: Fit the final classifier (e.g., LogisticRegression, SVC)
                    clf.fit(X_selected, y_balanced)
    
                    X_test_scaled = scaler.transform(X_test)
                    X_test_selected = X_test_scaled[:, mask]
                    y_pred = clf.predict(X_test_selected)
    
                    if hasattr(clf, "predict_proba"):
                        y_score = clf.predict_proba(X_test_selected)[:, 1]
                    elif hasattr(clf, "decision_function"):
                        y_score = clf.decision_function(X_test_selected)
                    else:
                        print(f"Warning: Classifier {clf.__name__} has neither predict_proba nor decision_function.")
                        y_score = y_pred
    
                    metrics = {
                        'f1_scores': [],
                        'conf_matrices': [],
                        'roc_aucs': [],
                        'g_means': [],
                        'proba_or_scores': [],
                        'y_trues': [],
                        'y_preds': [],
                        'clf_scores': [],
                        'feature_mask': [],
                    }
                
                    # Collect metrics for the current fold and percentile
                    metrics['y_trues'].append(y_test.values)
                    metrics['y_preds'].append(y_pred)
                    metrics['proba_or_scores'].append(y_score)
                    metrics['f1_scores'].append(f1_score(y_test, y_pred))
                    metrics['conf_matrices'].append(confusion_matrix(y_test, y_pred))
                    metrics['roc_aucs'].append(roc_auc_score(y_test, y_score))
                    metrics['g_means'].append(geometric_mean_score(y_test, y_pred))
                    metrics['clf_scores'].append(clf.score(X_test_selected, y_test))
                    metrics['feature_mask'].append(mask) 
    
                    name = "_".join([
                        cv_name,
                        pct_name,
                        fsmethod_name,
                        blmethod_name,
                        clmethod_name,
                        str(switch)
                    ])
                
                    filepath = f"./tmp/outv2_5/{country}"
                    os.makedirs(filepath, exist_ok=True)
                    joblib.dump(metrics, f"{filepath}/metrics-{name}.jbl")

In [ ]:
import shutil

# Zip the folder
shutil.make_archive('/content/drive/My Drive/outv2__', 'zip', '/content/drive/My Drive/outv2__')